<div style="background: linear-gradient(120deg, #1a3a5c 0%, #2d6a9f 60%, #4a9eda 100%); padding: 28px 36px; border-radius: 14px; display: flex; align-items: center; gap: 28px; box-shadow: 0 4px 18px rgba(0,0,0,0.18);">
    <img src='Figures/iteso.jpg' style='height: 110px; border-radius: 8px; background: white; padding: 6px; flex-shrink: 0;'/>
    <div style='border-left: 2px solid rgba(255,255,255,0.4); padding-left: 28px;'>
        <h1 style='margin: 0 0 8px 0; color: white; font-size: 1.5em;'>Ingeniería y Ciencia de Datos</h1>
        <h3 style='margin: 0 0 8px 0; color: white; font-size: 1.5em;'>Laboratorio de Procesamiento de Datos</h3>
        <h3 style='margin: 0; color: rgba(255,255,255,0.8); font-weight: normal; font-size: 1.05em;'>Módulo 2: Codificación de Variables Categóricas Nominales</h3>
    </div>
</div>


Una variable categórica guarda una etiqueta, no una cantidad. Para usarla en muchos modelos, la representamos con números. La codificación debe respetar la información que sí existe: dos categorías pueden ser distintas sin que una sea mayor que la otra.

En este cuaderno de jupyter veremos cómo representar una variable binaria con 0/1 y cómo extender esa idea a **variables categoricas nominales** con varias categorías. 

## Variables indicadoras y ejemplo

Una variable **binaria** tiene exactamente dos valores posibles y puede representarse con una sola columna 0/1.

El siguiente conjunto de datos relaciona una ciudad con una renta generada, estos datos son generados para mostrar las transformaciones.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression

In [ ]:
#Dataset de ejemplo
np.random.seed(1)
categorías=['Guadalajara','Zapopan','Ciudad de México','Monterrey','Morelia','Mexicali']
ponderación=[1/4,1/5,1/4,7/40,1/20,3/40]
limites=[(3500,6000),(3000,12000),(4000,18000),(3000,6000),(2000,5000),(3500,5000)]
ciudad=np.random.choice(categorías,size=20,p=ponderación)
rentas=[]
for c in ciudad:
    rentas.append(np.random.randint(*limites[categorías.index(c)]))
df=pd.DataFrame()
df['Rentas']=rentas
df['Ciudad']=ciudad
df.head()

# Codificación One-Hot

## One-hot encoding

Crea una columna indicadora por categoría. Cada observación recibe `1` en la columna que le corresponde y `0` en las demás. Por ejemplo, para `Ciudad ∈ {Guadalajara, Zapopan, Morelia}`, una fila de Zapopan queda como `(0, 1, 0)`.

Es útil para variables nominales porque no asigna un orden artificial. El número de columnas crece con la cardinalidad: si hay $K$ categorías, se crean $K$ columnas. Para modelos lineales con intercepto, la suma de las columnas siempre es 1, por lo que incluirlas todas introduce redundancia; la codificación ficticia es una alternativa cuando se desea una categoría de referencia.

In [ ]:
pd.get_dummies(df)                             # Codifica todas las variables categóricas

In [ ]:
pd.get_dummies(df['Ciudad'])                  # No agrega el prefijo

In [ ]:
pd.get_dummies(df,columns=['Ciudad'])         # Codifica solo 'Ciudad' (Agrega el prefijo)

In [ ]:
help(pd.get_dummies)

In [ ]:
OH=pd.get_dummies(df,prefix='',prefix_sep='') # Especificamos el prefijo
OH.head(2)

### One-hot encoding con Scikit-learn

`OneHotEncoder` de scikit-learn es útil cuando la codificación forma parte del preprocesamiento de un modelo. `fit` aprende las categorías disponibles en los datos de entrenamiento; `transform` usa ese mismo esquema para convertir los datos. Separar ambas operaciones ayuda a evitar inconsistencias entre entrenamiento y evaluación.

En el ejemplo siguiente usamos `sparse_output=False` para mostrar una matriz densa pequeña. Para conjuntos con muchas categorías conviene conservar la salida dispersa. En un flujo de producción, configura también `handle_unknown` para decidir qué hacer cuando aparezca una categoría no observada durante el ajuste.

In [ ]:
#from sklearn.preprocessing import OneHotEncoder 
from sklearn.preprocessing import OneHotEncoder

df_ejemplo = pd.DataFrame({'Color': ['Rojo', 'Azul', 'Verde', 'Azul', 'Rojo']})
encoder = OneHotEncoder(sparse_output=False)
onehot = encoder.fit_transform(df_ejemplo[['Color']])
df_onehot_sklearn = pd.DataFrame(onehot, columns=encoder.get_feature_names_out(['Color']))
print(df_onehot_sklearn)

In [ ]:
# Ejemplo: conjunto de aprobación de crédito
data = pd.read_csv("Data/credit_approval_uci.csv")
data.head()

In [ ]:
encoder = OneHotEncoder(sparse_output=False)

In [ ]:
vars_categorical = data.select_dtypes(include="O").columns.to_list()
vars_categorical

In [ ]:
encoder.fit(data[vars_categorical])

In [ ]:
encoder.categories_

In [ ]:
data_enc =  encoder.transform(data[vars_categorical])
data_enc

In [ ]:
encoder.get_feature_names_out()

In [ ]:
#Transformamos a un dataframe
data_enc = pd.DataFrame(data_enc)
# agregamos el nombre de las variables categoricas encodeadas
data_enc.columns = encoder.get_feature_names_out()
data_enc.head()

In [ ]:
data_transform = pd.concat([data, data_enc], axis=1)
# Drop variables categoricas
data_transform.drop(labels=vars_categorical, axis=1, inplace=True)
data_transform


### Ventajas y límites de one-hot

One-hot conserva la identidad de las categorías sin asignarles un orden. Es compatible con muchos modelos, incluidos modelos lineales y redes neuronales.

Cada categoría añade una columna. Cuando hay muchas categorías, la matriz puede ser grande y dispersa; considera la cardinalidad antes de elegir esta técnica. En modelos lineales con intercepto, incluir todas las columnas puede generar redundancia perfecta. La codificación ficticia elimina una columna de referencia para evitarla.

Los valores faltantes requieren una decisión explícita: imputarlos, tratarlos como una categoría propia o usar las opciones del codificador. Una fila con ceros en todas las columnas no identifica automáticamente un dato faltante; también podría representar una categoría desconocida o una categoría de referencia, según el método.

# Codificación Ficticia

La codificación ficticia conserva una columna menos que one-hot: una categoría se elige como **referencia** y queda representada por ceros en las columnas restantes. Para $K$ categorías genera $K-1$ columnas. `drop_first=True` omite una categoría según el orden interno de pandas; revisa ese orden o define la referencia de manera explícita si importa para la interpretación.

En una regresión lineal, el intercepto corresponde a la predicción para la categoría de referencia y cada coeficiente representa la diferencia respecto a ella. Para una variable binaria, basta una columna: el coeficiente compara la categoría codificada como 1 con la codificada como 0.

In [ ]:
F=pd.get_dummies(df,prefix='',prefix_sep='',drop_first=True)
F.head(2)

In [ ]:
df

In [ ]:
np.where(df['Ciudad']=='Ciudad de México')

In [ ]:
F.iloc[[ 9, 11, 15, 17], :]

In [ ]:
CR,=set(OH.columns)-set(F.columns)
CR

### Codificación ficticia con scikit-learn

`OneHotEncoder(drop='first')` aplica la misma idea de referencia. El codificador aprende las categorías al ajustarse con los datos de entrenamiento y reutiliza ese esquema al transformar nuevos datos. Esto resulta especialmente útil en pipelines y permite configurar el tratamiento de categorías no vistas.

In [ ]:
encoder = OneHotEncoder(sparse_output=False, drop='first')
vars_categorical = data.select_dtypes(include="O").columns.to_list()
encoder.fit(data[vars_categorical])

In [ ]:
encoder.categories_

In [ ]:
data_enc =  encoder.transform(data[vars_categorical])

In [ ]:
#Transformamos a un dataframe
data_enc = pd.DataFrame(data_enc)
data_enc.columns = encoder.get_feature_names_out()
data_transform = pd.concat([data, data_enc], axis=1)
data_transform.drop(labels=vars_categorical, axis=1, inplace=True)
data_transform

# Codificación de Efectos

En codificación de efectos también se omite una columna, pero la categoría de referencia se representa con `-1` en las columnas restantes. Así, cada coeficiente de un modelo lineal se interpreta como una desviación respecto a la media general; la predicción de la categoría de referencia se obtiene combinando el intercepto y los coeficientes.

Por ejemplo, con tres categorías y `C` como referencia, `A = (1, 0)`, `B = (0, 1)` y `C = (-1, -1)`. El vector de referencia hace que los coeficientes se interpreten en relación con la media general, en vez de comparar cada nivel con `C`.

Este ejemplo construye esa representación manualmente para hacer visible el cambio. En proyectos reales conviene usar una implementación probada y comprobar cuál es la categoría de referencia.

In [ ]:
E = pd.get_dummies(df, prefix='', prefix_sep='', drop_first=True).astype(int)
CDMX, = np.where(df['Ciudad'] == CR)
CDMX

In [ ]:
E.iloc[CDMX,1:]=-1
E.head(10)

In [ ]:
E.dtypes

In [ ]:
E=E.astype('int')
E.dtypes

In [ ]:
E.iloc[CDMX,1:]=-1
E.head(10)

# Regresión lineal

En una regresión lineal, una variable binaria codificada como 0/1 permite interpretar su coeficiente como la diferencia entre la predicción del grupo 1 y la del grupo 0, manteniendo las demás variables constantes.

Para una variable con varias categorías, el significado del intercepto y de los coeficientes depende del esquema: con referencia, cada coeficiente compara un nivel con esa referencia; con efectos, las diferencias se expresan respecto a la media general. La codificación cambia la parametrización y la interpretación, no la información original de las categorías.

### Reg lineal con OneHot

In [ ]:
OH.head()

In [ ]:
linOH=LinearRegression()
linOH.fit(OH.drop(columns='Rentas'),OH.Rentas)

In [ ]:
linOH.coef_

In [ ]:
linOH.intercept_

$$ \hat{y} = a1*x_1 + a2*x_2 + ... + a5*x_5 + a0$$

In [ ]:
promedios=df.groupby('Ciudad').mean()
promedios

In [ ]:
linOH.intercept_-promedios.mean()

In [ ]:
promedios-promedios.mean()

In [ ]:
linOH.coef_

**Para la codificación One-Hot, en una regresión lineal, la intersección es el promedio de los valores promedios de las rentas de cada categoría. Los coeficientes son el resultante de la resta entre el promedio de las categorías y el promedio general.**

### Reg lineal con Codificación Ficticia

In [ ]:
linF=LinearRegression()
linF.fit(F.drop(columns='Rentas'),F.Rentas)

In [ ]:
CR

In [ ]:
linF.intercept_-promedios.loc[CR]

In [ ]:
linF.intercept_

In [ ]:
promedios.loc[CR]

In [ ]:
promedios-promedios.loc[CR]

In [ ]:
linF.coef_

**En la codificación Ficticia, la intersección de la regresión lineal corresponde al promedio de la categoría de referencia. Los coeficientes son la diferencia entre el promedio de cada categoría y la intersección.**

### Reg lineal con Codificación de Efectos

In [ ]:
linE=LinearRegression()
linE.fit(E.drop(columns='Rentas'),E.Rentas)

In [ ]:
linOH.intercept_,linE.intercept_

In [ ]:
linOH.coef_[1:],linE.coef_

In [ ]:
linE.intercept_+linE.coef_

In [ ]:
promedios

In [ ]:
# Promedio de la Categoría de referencia
linE.intercept_-linE.coef_.sum()

**Para la codificación de Efectos, en una regresión lineal, la intersección es el promedio general y los coeficientes, la diferencia entre los promedios de cada variable y el promedio general (a esto se le conoce como efecto principal).**

### ¿Cuándo usar codificación de efectos?

En modelos lineales y análisis de varianza, esta codificación permite comparar los niveles con la media general. No es universalmente mejor que one-hot o la codificación ficticia: cambia la interpretación del intercepto y de los coeficientes, pero no añade información nueva. Elige el esquema según la pregunta que quieras responder y verifica cómo trata tu librería la categoría de referencia.

## Comparación de métodos de codificación

| Método | Columnas para $K$ categorías | Cómo se representa la referencia | Interpretación típica | Útil cuando... |
|---|---:|---|---|---|
| Binaria 0/1 | 1 | 0 representa una categoría y 1 la otra | Diferencia entre las dos categorías | La variable tiene exactamente dos niveles |
| One-hot | $K$ | No hay referencia omitida | Cada columna identifica un nivel | Se quiere conservar la identidad de categorías nominales |
| Ficticia | $K-1$ | Vector de ceros | Cada coeficiente compara con el nivel de referencia | Se interpretan coeficientes de un modelo lineal |
| Efectos | $K-1$ | -1 en todas las columnas | Coeficientes como desviaciones respecto a la media general | Interesa comparar niveles con el promedio global |

**Idea clave:** el mismo predictor puede producir matrices distintas según el esquema. Elige la codificación antes de interpretar los coeficientes y verifica siempre cuál es la categoría de referencia.

## Práctica de Laboratorio

Usa `Data/credit_approval_uci.csv`. El dataset contiene predictores `A1` a `A15` y una variable de respuesta `target`. Mantén `target` separado: no debe formar parte de las columnas que vas a codificar.

1. Revisa cardinalidad y faltantes. ¿Qué columnas son binarias y cuáles tienen más de dos categorías?
2. Codifica `A9` como binaria. Explica qué categoría decidiste representar con 1 y por qué.
3. Codifica `A6` con one-hot y con codificación ficticia. Compara el número de columnas y muestra algunas filas.
4. Explica cómo representaste los valores faltantes y cuál sería la categoría de referencia en el caso ficticio.
